# 05 — Reconstruct accuracy and motion evidence

## Question

Does a method reduce coordinate error while preserving supported displacement, amplitude, timing and anatomical sides under person-level uncertainty?

## Inputs

Saved development predictions from unchanged, filtering, practical and representation arms, plus separate target validity, visibility, timestamps, evaluation scales and group identifiers.

Use an explicit `STV2_CONFIG` JSON file and a unique run ID. The setup resolves relative artifact paths from the repository root. See the [execution guide](README.md), [development protocol](../../docs/studies/synthetic-training-v2/protocol.md) and [literature ledger](../../docs/studies/synthetic-training-v2/literature.md).

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

if not os.environ.get("STV2_CONFIG"):
    raise RuntimeError("Set STV2_CONFIG to an explicit study JSON configuration before execution.")
config_path = Path(os.environ["STV2_CONFIG"]).expanduser().resolve()
if not config_path.is_file():
    raise FileNotFoundError(f"Study configuration does not exist: {config_path}")
search_root = Path(os.environ.get("GAVD6_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next((path for path in (search_root, *search_root.parents)
                     if (path / "src/gavd6_sjepa").is_dir() and (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run inside the repository or set GAVD6_ROOT to its root.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ["STV2_CONFIG"] = str(config_path)
from gavd6_sjepa.research_directions.synthetic_training_v2.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training_v2.workflow import run_stage

cfg = RunConfig.load(os.environ["STV2_CONFIG"])
display({"run_id": cfg.run_id, "mode": cfg.mode, "device": cfg.device,
         "artifact_root": str(cfg.root), "confirmation": "closed"})
if cfg.mode == "fixture":
    print("CPU software fixture: method ordering is not empirical evidence.")

## Computation

Recompute per-window endpoints and balanced person-level summaries, paired cluster uncertainty and nuisance strata. Plot visible coordinate error against fixed-lag displacement error. Report each extractor and seed separately. Here amplitude is the RMS of demeaned horizontal left-minus-right ankle separation, normalized by the reference-box diagonal. Event timing measures its positive local maxima in seconds; these are operational projected-trajectory events, not heel strikes or clinical events.

`run_stage` implements the computation in the study modules. It checks prerequisite receipts and returns the saved result on an unchanged rerun; a changed configuration or code identity requires a new run ID.

In [ ]:
evaluation = run_stage(cfg, "evaluate", repo_root=PROJECT_ROOT)
display({key: evaluation[key] for key in
         ("evidence_status", "windows", "extractors", "seeds")})
display({"sampling_groups": evaluation["independent_people"],
         "group_kind": "analytic fixture IDs" if cfg.mode == "fixture" else "audited canonical people"})
display(evaluation["gates"])
display(Image(filename=str(cfg.root / "evaluation/accuracy-preservation.png")))

## Outputs and checks

Inspect `evaluation/per-window.csv`, `per-person-balanced-summary.csv`, `nuisance-strata.csv`, contrast artifacts, `gates.json` and the tradeoff image. The displayed window count is metric rows across methods; it is not the number of independent samples. Check endpoint support counts and missing metrics before comparing methods.

Stage receipts under `receipts/` record elapsed time and hashes of produced artifacts. Inspect the saved files for full diagnostics; the display above is deliberately brief.

## Interpretation

The current workflow leaves Gate B insufficient until repeated-seed preservation, clean-retention and calibrated-margin adjudication exist. Independent real temporal references are also pending. Paired intervals resample people with motions nested, condition on the fitted model, and exclude training/selection uncertainty. In fixtures they check arithmetic on analytic groups, not population uncertainty. Low latent loss, high rank or a clean-looking curve cannot establish useful gait measurement. No time warping is allowed during scoring.

A completed fixture checks software behavior. Scientific gates use `pass`, `fail` or `insufficient_evidence`; fixture success cannot make a scientific gate pass.

## Next gate

Proceed to [06 — optional gates](06_optional_gates.ipynb). A failed JEPA gate stops that expansion while a useful direct denoiser may proceed independently. Do not convert missing motion evidence into a passing gate.